# Advanced Structures & Memory Optimization (__slots__) (5+ Years Interview Guide)
Exhaustive revision guide to defaultdict, Counter, deque, namedtuple, memory profiling (sys.getsizeof), and __slots__ on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Memory Inspection**: Dedicated cell for `sys.getsizeof()` and `id()`.
- **Specialized Collections**: Dedicated cell for `defaultdict`, `Counter`, `deque`, `namedtuple`, and `OrderedDict`.
- **Low-Level Memory Optimization**: Dedicated cell for `__slots__ = ('attr1', 'attr2')` eliminating `__dict__` overhead.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import json
import re
import collections
from datetime import datetime, timedelta

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Memory Inspection: `sys.getsizeof()` and `id()`
**Explanation**: `id()` returns the unique memory address pointer. `sys.getsizeof()` measures shallow RAM consumption of Python objects in bytes.

**Syntax**: `sys.getsizeof(obj)` / `id(obj)`

In [2]:
tx_sample = transactions[0]
print('Memory size of single transaction dict:', sys.getsizeof(tx_sample), 'bytes')
print('Memory address (id) of transaction:', id(tx_sample))

Memory size of single transaction dict: 464 bytes
Memory address (id) of transaction: 2188089958336


### Specialized Collections: `Counter` & `defaultdict`
**Explanation**: `collections.Counter` tallies elements in $O(N)$ time. `defaultdict` automatically initializes missing keys on access.

**Syntax**: `Counter(cards)` / `defaultdict(list)`

In [3]:
card_counts = collections.Counter([t['card_type'] for t in transactions])
print('Card Distribution (Counter):', card_counts)

regional_spend = collections.defaultdict(float)
for t in transactions:
    regional_spend[t['region']] += float(t['transaction_amount'])
print('Regional Total Spend (defaultdict):', {k: round(v, 2) for k, v in regional_spend.items()})

Card Distribution (Counter): Counter({'Amex': 3801, 'MasterCard': 3768, 'Discover': 3723, 'Visa': 3708})
Traceback (most recent call last):
  File "C:\Users\DELL\investigate-pandas\scratch\execute_and_populate_outputs.py", line 34, in execute_notebook
    exec(code, exec_globals)
  File "<string>", line 6, in <module>
ValueError: could not convert string to float: ''


### High-Speed Queues & Tuples: `deque` and `namedtuple`
**Explanation**: `deque` provides $O(1)$ fast appends and pops from both ends (unlike list's $O(N)$ pops from left). `namedtuple` provides lightweight tuple memory with dot-notation field access.

**Syntax**: `collections.deque(maxlen=10)` / `collections.namedtuple('Tx', fields)`

In [4]:
TxRecord = collections.namedtuple('TxRecord', ['tx_id', 'amount', 'card'])
sample_named_tx = TxRecord(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount']), transactions[0]['card_type'])
print('namedtuple access:', sample_named_tx.tx_id, sample_named_tx.amount, sample_named_tx.card)

recent_tx_queue = collections.deque(maxlen=3)
for t in transactions[:5]:
    recent_tx_queue.append(t['transaction_id'])
print('Rolling deque (maxlen=3):', list(recent_tx_queue))

namedtuple access: TX110686 1216.33 Visa
Rolling deque (maxlen=3): ['TX108328', 'TX108563', 'TX107002']


### Memory Optimization with `__slots__`
**Explanation**: `__slots__` bypasses the default per-instance dynamic `__dict__`, storing attributes in a fixed-size C-struct array cutting object memory consumption by 60%-70%.

**Syntax**: `__slots__ = ('tx_id', 'amount', 'card')`

In [5]:
class StandardTx:
    def __init__(self, tx_id, amount):
        self.tx_id = tx_id
        self.amount = amount

class SlottedTx:
    __slots__ = ('tx_id', 'amount')
    def __init__(self, tx_id, amount):
        self.tx_id = tx_id
        self.amount = amount

std_inst = StandardTx('TX1', 100.0)
slot_inst = SlottedTx('TX1', 100.0)
print('Standard instance with __dict__ size:', sys.getsizeof(std_inst) + sys.getsizeof(std_inst.__dict__), 'bytes')
print('Slotted instance size:', sys.getsizeof(slot_inst), 'bytes (no __dict__ overhead)')

Standard instance with __dict__ size: 344 bytes
Slotted instance size: 48 bytes (no __dict__ overhead)


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Large-Scale Object Instantiation: Dict vs Slotted Dataclass vs Namedtuple
**Explanation**: Compare RAM consumption of storing 100,000 transaction objects in memory.

**Syntax**: `[SlottedTx(...) for _ in range(N)]`

In [6]:
n_items = 5000
slotted_objs = [SlottedTx(t['transaction_id'], float(t['transaction_amount'])) for t in transactions[:n_items]]
print(f'Memory for {n_items} slotted objects: {sum(sys.getsizeof(o) for o in slotted_objs) / 1024:.1f} KB')

Traceback (most recent call last):
  File "C:\Users\DELL\investigate-pandas\scratch\execute_and_populate_outputs.py", line 34, in execute_notebook
    exec(code, exec_globals)
  File "<string>", line 2, in <module>
ValueError: could not convert string to float: ''
